# 🔬 crdt-merge v0.3.0 — A100 Absolute Ceiling Test

Push every module to its breaking point on A100 (80 GB RAM).
8 test suites, progressive scales from 100K to 100M rows.

**Two merge paradigms tested:**
- `merge()` — overlay merge (second-argument-wins, like git merge remote→local)
- `LWW()` strategy — true CRDT commutative merge (timestamp wins regardless of order)

Both are correct by design. Results saved as JSON for permanent record.

In [ ]:
!pip install crdt-merge==0.3.0 psutil 2>&1 | tail -3
import crdt_merge
print(f"crdt-merge {crdt_merge.__version__} ready")
import psutil
print(f"RAM: {psutil.virtual_memory().total/(1024**3):.1f} GB")

In [ ]:
import time, gc, json, sys, tracemalloc, os, psutil
from contextlib import contextmanager
from collections import defaultdict

RESULTS = []

def record(suite, test, scale, metric, value, unit, notes=""):
    r = {"suite": suite, "test": test, "scale": scale,
         "metric": metric, "value": round(value, 4), "unit": unit, "notes": notes}
    RESULTS.append(r)
    print(f"  [{suite}] {test} @ {scale:,}: {metric}={value:.4f} {unit} {notes}")

@contextmanager
def mem_track():
    gc.collect()
    tracemalloc.start()
    baseline = tracemalloc.get_traced_memory()[0]
    yield lambda: (tracemalloc.get_traced_memory()[1] - baseline) / (1024*1024)
    tracemalloc.stop()

def gen_rows(n, prefix="a", fields=5):
    """Generate n rows with zero-padded IDs for sort stability."""
    return [dict(_id=f"{prefix}_{i:010d}", _ts=float(i),
                 **{f"f{j}": f"v_{prefix}_{i}_{j}" for j in range(fields)}) for i in range(n)]

def sorted_gen(n, prefix="a", fields=5):
    """Memory-efficient generator yielding rows in sorted _id order."""
    for i in range(n):
        yield dict(_id=f"{prefix}_{i:010d}", _ts=float(i),
                   **{f"f{j}": f"v_{prefix}_{i}_{j}" for j in range(fields)})

def gen_conflict_pair(n, overlap=0.5):
    """Generate two datasets with controlled overlap for conflict testing."""
    olap = int(n * overlap)
    uniq = n - olap
    a = [{"_id": f"r_{i:010d}", "_ts": float(i), "val": f"a_{i}",
          "score": i % 100, "tags": f"t{i%10}"} for i in range(n)]
    b = [{"_id": f"new_{i:010d}", "_ts": float(i + n), "val": f"b_{i}",
          "score": (i + 50) % 100, "tags": f"t{(i+5)%10}"} for i in range(uniq)]
    b += [{"_id": f"r_{i:010d}", "_ts": float(i + n), "val": f"b_c_{i}",
           "score": (i + 25) % 100, "tags": f"t{(i+3)%10}"} for i in range(olap)]
    return a, b

print(f"Infrastructure ready | RAM: {psutil.virtual_memory().total/(1024**3):.1f} GB")

In [ ]:
from crdt_merge import merge
import psutil

print("=" * 70)
print("SUITE 1: CORE merge() — Overlay Merge Progressive Scale")
print("  Design: merge(local, remote) = remote wins on key collision")
print("=" * 70)

for n in [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    print(f"\n  Available RAM: {avail_gb:.1f} GB")
    if avail_gb < 1.5:
        record("core", "merge_oom", n, "oom", 1, "bool", "stopped — low RAM")
        print(f"  LOW RAM — ceiling found at {n:,}")
        break
    # 50% overlap to test both insert + overwrite paths
    a = gen_rows(n, "a", 3)
    b = gen_rows(n // 2, "b", 3) + gen_rows(n // 2, "a", 3)
    with mem_track() as get_mem:
        t0 = time.time()
        result = merge(a, b, key="_id")
        elapsed = time.time() - t0
        peak = get_mem()
    record("core", "merge", n, "time_sec", elapsed, "s")
    record("core", "merge", n, "throughput", len(result) / elapsed, "rows/s")
    record("core", "merge", n, "peak_mb", peak, "MB")
    record("core", "merge", n, "output_rows", len(result), "rows")
    del a, b, result; gc.collect()

In [ ]:
from crdt_merge.strategies import MergeSchema, LWW, MaxWins, MinWins, UnionSet, Priority

print("=" * 70)
print("SUITE 2: STRATEGIES — Per-Column Resolve Throughput")
print("  Design: True CRDT — commutative, timestamp-aware")
print("=" * 70)

schema = MergeSchema(
    default=LWW(),
    score=MaxWins(),
    tags=UnionSet(separator=","),
    status=Priority(["draft", "review", "published"])
)

for n in [100_000, 500_000, 1_000_000, 5_000_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    if avail_gb < 1.5:
        print(f"  LOW RAM at {n:,} — stopping"); break
    a, b = gen_conflict_pair(n, overlap=1.0)
    with mem_track() as get_mem:
        t0 = time.time()
        for i in range(len(a)):
            schema.resolve_row(a[i], b[i], timestamp_col="_ts")
        elapsed = time.time() - t0
        peak = get_mem()
    record("strategies", "resolve_row", n, "throughput", n / elapsed, "rows/s")
    record("strategies", "resolve_row", n, "peak_mb", peak, "MB")
    del a, b; gc.collect()

In [ ]:
from crdt_merge.streaming import merge_sorted_stream, merge_stream

print("=" * 70)
print("SUITE 3A: merge_stream — Batched In-Memory Merge")
print("=" * 70)

for n in [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    if avail_gb < 2.0:
        print(f"  LOW RAM at {n:,} — stopping"); break
    with mem_track() as get_mem:
        t0 = time.time()
        total_rows = 0
        for batch in merge_stream(gen_rows(n, "a", 3), gen_rows(n, "b", 3),
                                   key="_id", batch_size=10000):
            total_rows += len(batch)
        elapsed = time.time() - t0
        peak = get_mem()
    record("merge_stream", "batched", n, "throughput", total_rows / elapsed, "rows/s")
    record("merge_stream", "batched", n, "peak_mb", peak, "MB")
    record("merge_stream", "batched", n, "output_rows", total_rows, "rows")
    gc.collect()

print()
print("=" * 70)
print("SUITE 3B: merge_sorted_stream — TRUE CONSTANT MEMORY")
print("  The crown jewel: O(batch_size) memory regardless of input size")
print("=" * 70)

for n in [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 50_000_000, 100_000_000]:
    print(f"\n  Scale: {n:,} rows each | RAM: {psutil.virtual_memory().available/(1024**3):.1f} GB avail")
    with mem_track() as get_mem:
        t0 = time.time()
        total_rows = 0
        for batch in merge_sorted_stream(sorted_gen(n, "a", 3), sorted_gen(n, "b", 3),
                                          key="_id", batch_size=10000):
            total_rows += len(batch)
        elapsed = time.time() - t0
        peak = get_mem()
    record("sorted_stream", "constant_mem", n, "throughput", total_rows / elapsed, "rows/s")
    record("sorted_stream", "constant_mem", n, "peak_mb", peak, "MB")
    record("sorted_stream", "constant_mem", n, "output_rows", total_rows, "rows")
    gc.collect()

In [ ]:
from crdt_merge.streaming import merge_sorted_stream

print("=" * 70)
print("SUITE 4: BATCH SIZE TUNING — Find Optimal batch_size at 1M")
print("=" * 70)

n = 1_000_000
for bs in [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]:
    with mem_track() as get_mem:
        t0 = time.time()
        total_rows = 0
        for batch in merge_sorted_stream(sorted_gen(n, "a", 3), sorted_gen(n, "b", 3),
                                          key="_id", batch_size=bs):
            total_rows += len(batch)
        elapsed = time.time() - t0
        peak = get_mem()
    record("batch_tune", f"bs_{bs}", n, "throughput", total_rows / elapsed, "rows/s")
    record("batch_tune", f"bs_{bs}", n, "peak_mb", peak, "MB")
    gc.collect()

In [ ]:
from crdt_merge.delta import compute_delta, apply_delta
import psutil

print("=" * 70)
print("SUITE 5: DELTA SYNC — Compute + Apply")
print("  50% overlap: half new rows, half updates")
print("=" * 70)

for n in [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    if avail_gb < 2.0:
        print(f"  LOW RAM at {n:,} — stopping"); break
    local = gen_rows(n, "a", 3)
    # remote = first half same IDs (updates), second half new IDs
    remote = gen_rows(n // 2, "a", 3) + gen_rows(n // 2, "b", 3)
    # Bump timestamps on remote so updates are newer
    for r in remote[:n // 2]:
        r["_ts"] += 1000.0
    with mem_track() as get_mem:
        t0 = time.time()
        delta = compute_delta(local, remote, key="_id")
        compute_t = time.time() - t0
        t0 = time.time()
        result = apply_delta(local, delta, key="_id")
        apply_t = time.time() - t0
        peak = get_mem()
    record("delta", "compute", n, "throughput", n / compute_t, "rows/s")
    record("delta", "compute", n, "time_sec", compute_t, "s")
    record("delta", "apply", n, "throughput", delta.size / apply_t if apply_t > 0 else 0, "rows/s")
    record("delta", "apply", n, "time_sec", apply_t, "s")
    record("delta", "total", n, "peak_mb", peak, "MB")
    record("delta", "total", n, "delta_size", delta.size, "rows", f"added={delta.added} modified={delta.modified}")
    del local, remote, delta, result; gc.collect()

In [ ]:
from crdt_merge import merge
from crdt_merge.verify import verify_crdt
from crdt_merge.strategies import MergeSchema, LWW
import random

print("=" * 70)
print("SUITE 6A: VERIFY — core merge() with SET equality")
print("  merge() is overlay (non-commutative on values), but commutative on KEYS")
print("=" * 70)

def gen_data():
    n = random.randint(50, 200)
    return [{"_id": f"r_{i:06d}", "_ts": float(random.random() * 1000),
             "val": f"v_{random.randint(0,999)}"} for i in range(n)]

def merge_fn(a, b):
    return merge(a, b, key="_id")

def set_eq(a, b):
    """Set equality on keys — correct test for overlay merge."""
    return set(r["_id"] for r in a) == set(r["_id"] for r in b)

for n_trials in [100, 500, 1000]:
    t0 = time.time()
    result = verify_crdt(merge_fn, gen_data, trials=n_trials, eq_fn=set_eq)
    elapsed = time.time() - t0
    c = "PASS" if result.commutativity.passed else "FAIL"
    a = "PASS" if result.associativity.passed else "FAIL"
    i = "PASS" if result.idempotency.passed else "FAIL"
    record("verify_core", "set_eq", n_trials, "commutativity", 1 if result.commutativity.passed else 0, "pass")
    record("verify_core", "set_eq", n_trials, "associativity", 1 if result.associativity.passed else 0, "pass")
    record("verify_core", "set_eq", n_trials, "idempotency", 1 if result.idempotency.passed else 0, "pass")
    record("verify_core", "set_eq", n_trials, "time_sec", elapsed, "s")
    print(f"  {n_trials} trials: C={c} A={a} I={i} in {elapsed:.1f}s")

print()
print("=" * 70)
print("SUITE 6B: VERIFY — LWW strategy with FULL VALUE equality")
print("  True CRDT: commutative on both keys AND values")
print("=" * 70)

schema = MergeSchema(default=LWW())

def schema_merge(a, b):
    a_dict = {r["_id"]: r for r in a}
    b_dict = {r["_id"]: r for r in b}
    result = []
    for k in sorted(set(a_dict.keys()) | set(b_dict.keys())):
        if k in a_dict and k in b_dict:
            result.append(schema.resolve_row(a_dict[k], b_dict[k], timestamp_col="_ts"))
        elif k in a_dict:
            result.append(a_dict[k])
        else:
            result.append(b_dict[k])
    return result

def val_eq(a, b):
    """Full value equality — correct test for LWW strategy."""
    return {r["_id"]: r for r in a} == {r["_id"]: r for r in b}

for n_trials in [100, 500, 1000]:
    t0 = time.time()
    result = verify_crdt(schema_merge, gen_data, trials=n_trials, eq_fn=val_eq)
    elapsed = time.time() - t0
    c = "PASS" if result.commutativity.passed else "FAIL"
    a = "PASS" if result.associativity.passed else "FAIL"
    i = "PASS" if result.idempotency.passed else "FAIL"
    record("verify_lww", "val_eq", n_trials, "commutativity", 1 if result.commutativity.passed else 0, "pass")
    record("verify_lww", "val_eq", n_trials, "associativity", 1 if result.associativity.passed else 0, "pass")
    record("verify_lww", "val_eq", n_trials, "idempotency", 1 if result.idempotency.passed else 0, "pass")
    record("verify_lww", "val_eq", n_trials, "time_sec", elapsed, "s")
    print(f"  {n_trials} trials: C={c} A={a} I={i} in {elapsed:.1f}s")

In [ ]:
from crdt_merge.strategies import MergeSchema, LWW
import itertools, psutil

print("=" * 70)
print("SUITE 7: MULTI-NODE CONVERGENCE — 5 Replicas, All Permutations")
print("  Proves: any merge order → identical result (CRDT guarantee)")
print("=" * 70)

schema = MergeSchema(default=LWW())

def schema_merge(a, b):
    a_dict = {r["_id"]: r for r in a}
    b_dict = {r["_id"]: r for r in b}
    result = []
    for k in sorted(set(a_dict.keys()) | set(b_dict.keys())):
        if k in a_dict and k in b_dict:
            result.append(schema.resolve_row(a_dict[k], b_dict[k], timestamp_col="_ts"))
        elif k in a_dict:
            result.append(a_dict[k])
        else:
            result.append(b_dict[k])
    return result

for n in [1_000, 10_000, 50_000, 100_000, 500_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    if avail_gb < 2.0:
        print(f"  LOW RAM at {n:,} — stopping"); break
    num_replicas = 5
    replicas = []
    for r in range(num_replicas):
        data = [{"_id": f"r_{i:08d}", "_ts": float(i + r * 0.1),
                 "val": f"node{r}_{i}", "score": (i + r * 17) % 100}
                for i in range(n)]
        replicas.append(data)

    t0 = time.time()
    # Test up to 10 random permutations of merge order
    results = []
    perms = list(itertools.permutations(range(num_replicas)))[:10]
    for perm in perms:
        merged = replicas[perm[0]]
        for idx in perm[1:]:
            merged = schema_merge(merged, replicas[idx])
        results.append({r["_id"]: r for r in merged})

    converged = all(r == results[0] for r in results[1:])
    elapsed = time.time() - t0
    record("convergence", f"{num_replicas}_replicas", n, "converged", 1 if converged else 0, "bool")
    record("convergence", f"{num_replicas}_replicas", n, "time_sec", elapsed, "s")
    record("convergence", f"{num_replicas}_replicas", n, "perms_tested", len(perms), "count")
    status = "✅ CONVERGED" if converged else "❌ DIVERGED"
    print(f"  {n:>10,} rows × {num_replicas} replicas × {len(perms)} perms: {status} ({elapsed:.1f}s)")
    del replicas, results; gc.collect()

In [ ]:
from crdt_merge import merge
from crdt_merge.streaming import merge_sorted_stream
import psutil

print("=" * 70)
print("SUITE 8: MEMORY SCALING — O(n) vs O(batch_size)")
print("  The definitive proof that streaming unlocks unlimited scale")
print("=" * 70)

print("\n--- merge() → O(n) memory (grows with input) ---")
for n in [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    avail_gb = psutil.virtual_memory().available / (1024**3)
    if avail_gb < 1.5:
        record("memory", "merge_On_oom", n, "oom", 1, "bool", "RAM ceiling hit")
        print(f"  {n:>12,}: OOM — ceiling found")
        break
    with mem_track() as get_mem:
        result = merge(gen_rows(n, "a", 3), gen_rows(n, "b", 3), key="_id")
        peak = get_mem()
    record("memory", "merge_On", n, "peak_mb", peak, "MB")
    print(f"  {n:>12,}: {peak:>8.1f} MB")
    del result; gc.collect()

print("\n--- merge_sorted_stream() → O(batch_size) memory (FLAT) ---")
for n in [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 50_000_000, 100_000_000]:
    print(f"  {n:>12,}: ", end="", flush=True)
    with mem_track() as get_mem:
        total = 0
        for batch in merge_sorted_stream(sorted_gen(n, "a", 3), sorted_gen(n, "b", 3),
                                          key="_id", batch_size=10000):
            total += len(batch)
        peak = get_mem()
    record("memory", "sorted_stream_Obatch", n, "peak_mb", peak, "MB")
    record("memory", "sorted_stream_Obatch", n, "output_rows", total, "rows")
    print(f"{peak:>8.1f} MB ({total:>13,} rows)")
    gc.collect()

In [ ]:
import json
from datetime import datetime
import psutil

report = {
    "version": "0.3.0",
    "date": datetime.now().isoformat(),
    "platform": "Google Colab A100",
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 1),
    "total_measurements": len(RESULTS),
    "results": RESULTS
}

with open("crdt_merge_v030_a100_results.json", "w") as f:
    json.dump(report, f, indent=2)

print("=" * 70)
print(f"  CRDT-MERGE v0.3.0 — A100 STRESS TEST COMPLETE")
print(f"  {len(RESULTS)} total measurements")
print(f"  Platform: Colab A100, {report['ram_gb']} GB RAM")
print("=" * 70)

suites = defaultdict(list)
for r in RESULTS:
    suites[r["suite"]].append(r)

for suite, entries in suites.items():
    print(f"\n{'─'*50}")
    print(f"  {suite.upper()}")
    print(f"{'─'*50}")
    for e in entries:
        if e["metric"] == "throughput":
            print(f"  {e['test']:>25s} @ {e['scale']:>12,}: {e['value']:>12,.0f} {e['unit']}")
        elif e["metric"] == "peak_mb":
            print(f"  {e['test']:>25s} @ {e['scale']:>12,}: {e['value']:>8.1f} MB")
        elif e["metric"] in ("converged", "commutativity", "associativity", "idempotency"):
            status = "✅" if e["value"] == 1 else "❌"
            print(f"  {e['test']:>25s} @ {e['scale']:>12,}: {e['metric']}={status}")

print(f"\n{'='*70}")
print(f"  JSON saved: crdt_merge_v030_a100_results.json")
print(f"  Share this file for permanent record in the repo")
print(f"{'='*70}")